# 🚀 Generar texto con GPT-2 y resumir con T5
*Curso de NLP · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** ver cómo cambian los textos de **GPT-2** según la estrategia de generación, y **ajustar T5** para que
escriba resúmenes, medidos con **ROUGE**.

**Modelos:** GPT-2 (124 millones de parámetros, MIT) y T5-small (60 millones, Apache 2.0). **Datos:** CNN/DailyMail
(`abisee/cnn_dailymail`, Apache 2.0): noticias en inglés con su resumen escrito por los periodistas.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")

In [ ]:
from pathlib import Path
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/MiAtlas-NLP")
else:
    CARPETA = Path("MiAtlas-NLP")
CARPETA.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", CARPETA)

In [ ]:
%pip install -q transformers datasets evaluate rouge_score accelerate

## 1. GPT-2: cuatro maneras de elegir la siguiente palabra

| Estrategia | Cómo elige | Resultado típico |
|---|---|---|
| Codiciosa (*greedy*) | Siempre la más probable | Correcta, pero se repite |
| Haz (*beam search*) | Varias secuencias a la vez, la más probable entera | Más coherente, poco variada |
| Muestreo con temperatura | Al azar, según la probabilidad (temperatura = cuánto se aplana) | Variada, a veces sin sentido |
| *Top-k* / *top-p* | Al azar, pero solo entre las k mejores o las que suman p | El equilibrio usual |

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

gpt_tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(DISPOSITIVO)
entrada = gpt_tok("The future of artificial intelligence in education", return_tensors="pt").to(DISPOSITIVO)

estrategias = {
    "codiciosa": dict(do_sample=False),
    "haz (5)": dict(num_beams=5, no_repeat_ngram_size=2, early_stopping=True),
    "temperatura 1.2": dict(do_sample=True, temperature=1.2),
    "top-p 0.9": dict(do_sample=True, top_p=0.9, top_k=0),
}
for nombre, opciones in estrategias.items():
    set_seed(0)
    salida = gpt.generate(**entrada, max_new_tokens=40, pad_token_id=gpt_tok.eos_token_id, **opciones)
    print(f"── {nombre}\n{gpt_tok.decode(salida[0], skip_special_tokens=True)}\n")

## 2. T5: resumir es «traducir» de texto largo a texto corto

T5 trata todo como texto a texto: basta anteponer `summarize:`.

In [ ]:
from datasets import load_dataset

noticias = load_dataset("abisee/cnn_dailymail", "3.0.0")
N_TRAIN, N_VAL = (200, 20) if PRUEBA_RAPIDA else (8000, 300)
train = noticias["train"].shuffle(seed=0).select(range(N_TRAIN))
val = noticias["validation"].shuffle(seed=0).select(range(N_VAL))
print(train[0]["article"][:400], "...\n\nRESUMEN:", train[0]["highlights"])

In [ ]:
from transformers import AutoModelForSeq2SeqLM

t5_tok = AutoTokenizer.from_pretrained("google-t5/t5-small")
t5 = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-small").to(DISPOSITIVO)

def preparar(lote):
    entradas = t5_tok(["summarize: " + a for a in lote["article"]], max_length=512, truncation=True)
    entradas["labels"] = t5_tok(text_target=lote["highlights"], max_length=128, truncation=True)["input_ids"]
    return entradas

tok_train = train.map(preparar, batched=True, remove_columns=train.column_names)
tok_val = val.map(preparar, batched=True, remove_columns=val.column_names)

In [ ]:
import evaluate
rouge = evaluate.load("rouge")

def resumir(articulos, modelo):
    entradas = t5_tok(["summarize: " + a for a in articulos], max_length=512, truncation=True,
                      padding=True, return_tensors="pt").to(DISPOSITIVO)
    salidas = modelo.generate(**entradas, max_new_tokens=80, num_beams=4)
    return t5_tok.batch_decode(salidas, skip_special_tokens=True)

def medir_rouge(modelo, n=100):
    muestra = val.select(range(min(n, len(val))))
    pred = []
    for i in range(0, len(muestra), 16):
        pred += resumir(muestra[i:i + 16]["article"], modelo)
    return rouge.compute(predictions=pred, references=muestra["highlights"])

antes = medir_rouge(t5, n=20 if PRUEBA_RAPIDA else 100)
print("ROUGE antes de ajustar:", {k: round(v, 3) for k, v in antes.items()})

**ROUGE** compara las palabras del resumen generado con las del resumen humano: **ROUGE-1** (palabras sueltas),
**ROUGE-2** (pares) y **ROUGE-L** (la secuencia común más larga). Va de 0 a 1: más alto es mejor.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

argumentos = Seq2SeqTrainingArguments(
    output_dir="t5-resumenes", report_to="none", save_strategy="no",
    learning_rate=3e-4, per_device_train_batch_size=8, num_train_epochs=1, fp16=False,
)
entrenador = Seq2SeqTrainer(model=t5, args=argumentos, train_dataset=tok_train,
                            data_collator=DataCollatorForSeq2Seq(t5_tok, model=t5))
entrenador.train()
despues = medir_rouge(t5, n=20 if PRUEBA_RAPIDA else 100)
print("ROUGE después de ajustar:", {k: round(v, 3) for k, v in despues.items()})

In [ ]:
ejemplo = val[0]
print("NOTICIA:", ejemplo["article"][:500], "...\n")
print("RESUMEN HUMANO:", ejemplo["highlights"], "\n")
print("RESUMEN DE T5:", resumir([ejemplo["article"]], t5)[0])

## Pruébalo tú

Escribe `generar(texto, top_p)`, que devuelva 30 palabras nuevas de GPT-2 con muestreo *top-p*. Pruébala con
`top_p=0.5` y `top_p=0.95`: ¿cuál es más creativa?

In [ ]:
def generar(texto, top_p):
    # ← tu código (pista: el diccionario «estrategias» de la sección 1)
    return None

In [ ]:
# Comprobación
r = generar("Once upon a time", 0.9)
if r is None:
    print("Aún no lo resuelves.")
elif isinstance(r, str) and r.startswith("Once upon a time"):
    print("✅ Funciona:", r[:120])
else:
    print("❌ Debe devolver un texto que empiece con la entrada")

## Resumen

- La **estrategia de generación** cambia el texto tanto como el modelo.
- T5 resume con el prefijo `summarize:` y mejora con un ajuste fino corto.
- **ROUGE** mide el parecido con un resumen humano, no si el resumen es verdadero.

**Fuentes:** [Transformers · estrategias de generación](https://huggingface.co/docs/transformers/generation_strategies) · [Transformers · resúmenes](https://huggingface.co/docs/transformers/tasks/summarization) · [GPT-2 (MIT)](https://huggingface.co/openai-community/gpt2) · [T5-small (Apache 2.0)](https://huggingface.co/google-t5/t5-small) · [CNN/DailyMail](https://huggingface.co/datasets/abisee/cnn_dailymail) · [ROUGE](https://huggingface.co/spaces/evaluate-metric/rouge)